In [1]:
import requests
import xml.etree.ElementTree as ET
from datetime import date

In [2]:
# ЦБ РФ отдаёт данные в формате XML
CBR_URL = "https://www.cbr.ru/scripts/XML_daily.asp"

In [ ]:
def fetch_rates(on_date: date = None) -> str:
    """Скачать XML с курсами валют с сайта ЦБ РФ"""
    params = {}
    if on_date:
        params["date_req"] = on_date.strftime("%d/%m/%Y")

    response = requests.get(CBR_URL, params=params, timeout=10)
    response.raise_for_status()
    response.encoding = "windows-1251"
    return response.text

def parse_rates(xml_text: str) -> list[dict]:
    """Распарсить XML и вернуть список словарей"""
    root = ET.fromstring(xml_text)

    # Дата из атрибута корневого элемента
    raw_date = root.attrib.get("Date", "")
    rate_date = date.today()
    if raw_date:
        from datetime import datetime
        rate_date = datetime.strptime(raw_date, "%d.%m.%Y").date()

    rates = []
    for valute in root.findall("Valute"):
        rates.append({
            "char_code": valute.find("CharCode").text,
            "name": valute.find("Name").text,
            "nominal": int(valute.find("Nominal").text),
            "rate": float(
                valute.find("Value").text.replace(",", ".")
            ),
            "date": rate_date,
        })

    return rates

In [15]:
def filter_currencies(
    rates: list[dict],
    currencies: list[str] = None
) -> list[dict]:
    """Оставить только нужные валюты"""
    if currencies is None:
        currencies = ["USD", "EUR", "CNY", "GBP"]

    return [r for r in rates if r["char_code"] in currencies]


def calculate_cross_rate(
    rates: list[dict],
    from_code: str,
    to_code: str
) -> float | None:
    """Посчитать кросс-курс между двумя валютами"""
    rate_map = {r["char_code"]: r["rate"] / r["nominal"] for r in rates}

    if from_code not in rate_map or to_code not in rate_map:
        return None

    return round(rate_map[from_code] / rate_map[to_code], 6)

In [17]:
# Extract — загрузить данные
print("📥 Загружаем данные с ЦБ РФ...")
xml_text = fetch_rates()
all_rates = parse_rates(xml_text)
print(f"   Получено валют: {len(all_rates)}")

# Transform — обработать данные
print("⚙️  Фильтруем валюты...")
rates = filter_currencies(all_rates, ["USD", "EUR", "CNY", "GBP", "JPY"])
print(f"   Оставлено валют: {len(rates)}")
for r in rates:
    print(f"   {r['char_code']}: {r['rate']} руб.")

📥 Загружаем данные с ЦБ РФ...
   Получено валют: 54
⚙️  Фильтруем валюты...
   Оставлено валют: 5
   GBP: 96.3707 руб.
   USD: 71.9077 руб.
   EUR: 82.9743 руб.
   CNY: 10.6064 руб.
   JPY: 44.7911 руб.


In [18]:
rates

[{'char_code': 'GBP',
  'name': 'Фунт стерлингов',
  'nominal': 1,
  'rate': 96.3707,
  'date': datetime.date(2026, 6, 12)},
 {'char_code': 'USD',
  'name': 'Доллар США',
  'nominal': 1,
  'rate': 71.9077,
  'date': datetime.date(2026, 6, 12)},
 {'char_code': 'EUR',
  'name': 'Евро',
  'nominal': 1,
  'rate': 82.9743,
  'date': datetime.date(2026, 6, 12)},
 {'char_code': 'CNY',
  'name': 'Юань',
  'nominal': 1,
  'rate': 10.6064,
  'date': datetime.date(2026, 6, 12)},
 {'char_code': 'JPY',
  'name': 'Иен',
  'nominal': 100,
  'rate': 44.7911,
  'date': datetime.date(2026, 6, 12)}]